# Modules over the triplet algebra $\mathcal W(2)$

The triplet algebras $\mathcal W(p)$ [Kau1991, GK1996b] are the standard examples of vertex algebras that are $C_2$-cofinite but not rational [Abe2007, AM2008]: they have finitely many simple modules, but also indecomposable modules on which $L_0$ is not diagonalisable, the *logarithmic modules*. For $p = 2$, $\mathcal W(2)$ is the even part of a pair of symplectic fermions $\chi^\pm$, of central charge $c = -2$ [Kau2000, Abe2007]. It has four simple modules $\mathcal W(h)$, of lowest weights $h = 0, 1, -\frac18, \frac38$. The modules $\mathcal W(-\frac18)$ and $\mathcal W(\frac38)$ are projective, while the projective cover of the vacuum module has the socle series structure [GK1996b]

$$\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0),$$

in which the arrows go from the head, through the middle layer, to the socle.

This notebook computes all of this from the definition of $\mathcal W(2)$ as a subalgebra, with general constructions of the library. Each result is checked against the symplectic fermion Fock spaces, character by character in the depth and the fermion charge: the untwisted sector gives $\mathcal W(0)$ and $\mathcal W(1)$ (its even and odd parts), and the $\mathbb Z_2$-twisted sector gives $\mathcal W(-\frac18)$ and $\mathcal W(\frac38)$.

**Prerequisites:** `06_subalgebras_ideals_quotients.ipynb`, `09_zhu_algebras.ipynb`, `10_modules.ipynb`, `11_fusion_and_extensions.ipynb`.

**Elsewhere:** the $(p_+, p_-)$ triplet algebras, as subalgebras of lattice vertex algebras, are in `14_triplet_algebra.ipynb`.

**Contents**

1. $\mathcal W(2)$ inside symplectic fermions (`V.subalgebra`)
2. The presentation (`S.presentation()`): OPEs that satisfy the Jacobi identities only modulo null fields
3. The Zhu algebra and its modules (`P.zhu_algebra_up_to(W)`): the four simple top spaces and a Jordan block
4. The simple modules, as universal modules $\bar M(U)$ (`P.induced_module_up_to(U, max_weight, max_depth)`) and their simple quotients
5. The module $\mathcal W(0) \to 2\,\mathcal W(1)$, induced from the trivial top space
6. The projective cover $\mathcal P(0)$, induced from the Jordan block of $L_0$
7. Up to a weight and a depth: the stability of the results
8. The projective cover $\mathcal P(1)$ through the level-1 Zhu algebra
9. Fusion products
10. The same, by the library: simple modules, projective covers, Loewy layers, extensions
11. Limitations

In [1]:
import vertex_algebras as va
from vertex_algebras import *
from vertex_algebras import UniversalVertexAlgebra, NormallyOrderedExpressions

## 1. $\mathcal W(2)$ inside symplectic fermions

Symplectic fermions are a pair of odd fields $\chi^\pm$ of weight 1 with $\chi^+(z)\chi^-(w) \sim (z-w)^{-2}$, with the conformal vector $T = {:}\chi^-\chi^+{:}$ of central charge $c = -2$, and graded by the charge, $\chi^\pm$ having charge $\pm1$ [Kau2000]. The triplet algebra is generated by $T$ and the fields of weight 3

$$W^+ = {:}\chi^+\partial\chi^+{:}, \qquad W^0 = {:}\chi^+\partial\chi^-{:} + {:}\chi^-\partial\chi^+{:}, \qquad W^- = {:}\chi^-\partial\chi^-{:},$$

of charges $2, 0, -2$. The cell below builds the symplectic fermions and the subalgebra generated by these four fields (`06_subalgebras_ideals_quotients.ipynb`, section 1); the closure finds no further strong generators. The cell prints the strong generators, their conformal weights and the central charge.

In [2]:
Y.<cp, cm> = NormallyOrderedExpressions(QQ, parities={'cp': 1, 'cm': 1})
SF = UniversalVertexAlgebra(QQ, Y, {('cp', 'cm'): [0, 1]}, conformal_vector=cm*cp,
                            horizontal_grading={'cp': 1, 'cm': -1})
a, b = SF.gens()
Wp, Wm, W0 = a.no(a.derivative()), b.no(b.derivative()), a.no(b.derivative()) + b.no(a.derivative())
W2 = SF.subalgebra([SF.conformal_vector(), Wp, W0, Wm], names=['T', 'Wp', 'W0', 'Wm'],
                   horizontal_grading='ambient', conformal_vector='ambient')
print(W2.variable_names(), [W2.degree(g) for g in W2.gens()], W2.central_charge())

('T', 'Wp', 'W0', 'Wm') [2, 3, 3, 3] -2


The characters of the four simple modules come from the Fock spaces of the fermion modes: $\prod_{n \ge 1}(1 + y q^n)(1 + y^{-1} q^n)$ for the untwisted sector, with the modes $\chi^\pm_{-n}$, and $\prod_{n \ge 0}(1 + y q^{n+1/2})(1 + y^{-1} q^{n+1/2})$ for the twisted sector, with the modes $\chi^\pm_{-n-1/2}$, where $y$ counts the fermion charge. Each simple module for the triplet algebra $\mathcal W(2)$ is the even or the odd part of one of the two sectors, where the parity is the fermion number modulo 2, that is, the charge modulo 2. In the cell below, `chi(h, d, g)` is the dimension of $\mathcal W(h)$ at depth $d$ (weight $h + d$) and charge $g$. The cell checks that the vacuum module $\mathcal W(0)$ is the VOA $\mathcal W(2)$ itself, by comparing the dimensions of the weight spaces up to weight 8.

In [3]:
R.<y> = LaurentPolynomialRing(QQ)
S.<s> = PowerSeriesRing(R, default_prec=20)          # s = q^(1/2)
untwisted = prod((1 + y*s^(2*n)) * (1 + y^-1*s^(2*n)) for n in range(1, 10))
twisted = prod((1 + y*s^(2*n+1)) * (1 + y^-1*s^(2*n+1)) for n in range(0, 10))


def chi(h, d, g):
    """The dimension of the simple module W(h) at depth d and charge g."""
    F, k, parity = {0: (untwisted, 2*d, 0), 1: (untwisted, 2*d + 2, 1),
                    -1/8: (twisted, 2*d, 0), 3/8: (twisted, 2*d + 1, 1)}[h]
    return F[k][g] if (g - parity) % 2 == 0 else 0


charges = range(-6, 7)
print([W2.weight_space(d).dimension() for d in range(9)])
print([sum(chi(0, d, g) for g in charges) for d in range(9)])

[1, 0, 1, 4, 5, 8, 10, 16, 22]
[1, 0, 1, 4, 5, 8, 10, 16, 22]


## 2. The presentation

$\mathcal W(2)$ is strongly generated by $T, W^\pm, W^0$, but not freely. `W2.presentation()` writes it as a quotient $F/R$ (`06_subalgebras_ideals_quotients.ipynb`, section 6): $F$ is the universal vertex algebra on these four fields, with their OPEs written in their normally ordered monomials, and $R$ is the kernel of the map $F \to \mathcal W(2)$. Here $F$ itself does not satisfy the Jacobi identities (`12_ope_bootstrap.ipynb`, section 1): the failures are null fields, and all of them lie in $R$. The cell below counts the failures and checks that they lie in $R$. It then prints the relation of weight 6 and charge 4, ${:}W^+W^+{:} = 0$, and the numbers of relations in the weights $6, 7, 8$, charge by charge; the first relations are at weight 6.

In [4]:
P = W2.presentation(); F = P.ambient()
fails = F.jacobi_identity_failures()
print(len(fails), all(x in P.defining_ideal() for x in fails.values()))
print(P.relations(6, horizontal_grade=4))
print({w: {g: len(P.relations(w, horizontal_grade=g)) for g in range(-6, 7, 2) if P.relations(w, horizontal_grade=g)}
       for w in range(6, 9)})

66 True
[:Wp Wp:]
{6: {-4: 1, -2: 2, 0: 3, 2: 2, 4: 1}, 7: {-4: 1, -2: 3, 0: 4, 2: 3, 4: 1}, 8: {-4: 3, -2: 6, 0: 9, 2: 6, 4: 3}}


## 3. The Zhu algebra and its modules

The Zhu algebra of the presentation, `P.zhu_algebra_up_to(W)`, is the free algebra on $[T], [W^+], [W^0], [W^-]$ modulo the commutator relations and the images of the relations of weight $\le W$, computed with two-sided Gröbner bases (Letterplace; see `09_zhu_algebras.ipynb`, section 6). It maps onto the Zhu algebra $A(\mathcal W(2))$ [Zhu1996]. The first line printed by the cell below shows that its dimension stabilises at 11 from $W = 7$ on, which matches the dimension of $A(\mathcal W(2))$ found by Adamović and Milas [AM2008]. The image $[\omega] = [T]$ of the conformal vector has the minimal polynomial $x^2(x - 1)(x + \frac18)(x - \frac38)$, whose roots are the four lowest weights; the root $0$ has multiplicity 2.

In [5]:
print([(W, P.zhu_algebra_up_to(W).dimension()) for W in range(5, 10)])
Z = P.zhu_algebra_up_to(8)
x = polygen(QQ, 'x')


def minimal_polynomial(Z, z, bound=12):
    """The minimal polynomial of the element z of the finite-dimensional algebra Z."""
    powers = [Z.one()]
    for k in range(1, bound + 1):
        powers.append(powers[-1] * z)
        keys = sorted({w for p in powers for w in p.support()})
        K = matrix(QQ, [[p.monomial_coefficients().get(w, 0) for w in keys] for p in powers]).left_kernel()
        if K.dimension():
            return sum(c * x^i for i, c in enumerate(K.basis()[0])).monic()


print(Z.gens(), minimal_polynomial(Z, Z.gen(0)).factor())

[(5, +Infinity), (6, 15), (7, 11), (8, 11), (9, 11)]
([T], [Wp], [W0], [Wm]) (x - 1) * (x - 3/8) * (x + 1/8) * x^2


The modules of the Zhu algebra are the possible top spaces (`10_modules.ipynb`, section 3). The Gröbner basis of the Zhu algebra is available as `Z.groebner_basis()`, in the free algebra on $x_0, \dots, x_3$, which stand for $[T], [W^+], [W^0], [W^-]$. So small modules can be found by evaluating the Gröbner basis on matrices with unknown entries and solving the resulting polynomial equations. The cell below does this twice:

* for one-dimensional modules of charge 0, on which $[W^\pm]$ act by 0: the solutions are $h = 0$ and $h = -\frac18$;
* for two-dimensional modules on two vectors of charges $\pm 1$, on which $[W^+]$ raises and $[W^-]$ lowers the charge by 2: besides sums of one-dimensional modules, the solutions are the doublets at $h = 1$ and $h = \frac38$ (the last two of the components printed).

In [6]:
G = Z.groebner_basis()


def evaluate(f, mats):
    """The element f of the free algebra evaluated on the matrices mats (x_i -> mats[i])."""
    out = 0 * mats[0]
    for m, c in f.monomial_coefficients().items():
        p = mats[0].parent().identity_matrix()
        for v, e in m:
            p = p * mats[int(str(v)[1:])]^e
        out += c * p
    return out


R1.<t, w> = QQ[]
print(R1.ideal([evaluate(f, [matrix(R1, [[t]]), matrix(R1, [[0]]), matrix(R1, [[w]]), matrix(R1, [[0]])])[0, 0]
                for f in G]).variety())
R2.<p, m, c, d, t1, t2> = QQ[]
mats = [matrix(R2, [[t1, 0], [0, t2]]), matrix(R2, [[0, p], [0, 0]]), matrix(R2, [[c, 0], [0, d]]),
        matrix(R2, [[0, 0], [m, 0]])]
I2 = R2.ideal([e for f in G for e in evaluate(f, mats).list()])
for Q in I2.minimal_associated_primes():
    print(Q.gens())

[{w: 0, t: 0}, {w: 0, t: -1/8}]
[p, m, c, d, t1, t2]
[t2, 8*t1 + 1, d, c, m, p]
[8*t2 + 1, t1, d, c, m, p]
[8*t2 + 1, 8*t1 + 1, d, c, m, p]
[p*m + 4, t2 - 1, t1 - 1, d + 2, c - 2]
[4*p*m + 1, 8*t2 - 3, 8*t1 - 3, 2*d + 1, 2*c - 1]


So the simple top spaces are the following, with $[W^+]$ normalised to $\begin{pmatrix} 0 & 1 \\ 0 & 0\end{pmatrix}$ on the doublets. Their dimensions $1, 1, 2, 2$ account for the dimension of the Zhu algebra: $11 = 1 + 1 + 4 + 4 + 1$ is the sum of their squares plus 1 for the nilpotent part of $[T]$ at $h = 0$, which gives a $2 \times 2$ Jordan block. The cell below builds the four simple top spaces and the Jordan block as modules over the Zhu algebra, graded by the charges (`horizontal_grades=`).

In [7]:
Z0 = matrix(QQ, 2, 2)
tops = {
    0: Z.module({'T': 0, 'Wp': 0, 'W0': 0, 'Wm': 0}, horizontal_grades=[0]),
    -1/8: Z.module({'T': -1/8, 'Wp': 0, 'W0': 0, 'Wm': 0}, horizontal_grades=[0]),
    1: Z.module({'T': identity_matrix(QQ, 2), 'Wp': matrix([[0, 1], [0, 0]]), 'W0': diagonal_matrix([2, -2]),
                 'Wm': matrix([[0, 0], [-4, 0]])}, horizontal_grades=[1, -1]),
    3/8: Z.module({'T': 3/8*identity_matrix(QQ, 2), 'Wp': matrix([[0, 1], [0, 0]]),
                   'W0': diagonal_matrix([1/2, -1/2]), 'Wm': matrix([[0, 0], [-1/4, 0]])}, horizontal_grades=[1, -1]),
}
jordan = Z.module({'T': matrix([[0, 1], [0, 0]]), 'Wp': Z0, 'W0': Z0, 'Wm': Z0}, horizontal_grades=[0, 0])
print(Z.dimension(), [U.dimension() for U in tops.values()], jordan.dimension())

11 [1, 1, 2, 2] 2


## 4. The simple modules

`P.induced_module_up_to(U, max_weight, max_depth)` is the universal module $\bar M(U)$ of Dong, Li and Mason with top space $U$ [DLM1998] (`10_modules.ipynb`, section 8). It is computed as the module induced over $F$, modulo the subspace generated by the relations of weight $\le$ `max_weight` and by the defects of the commutator formula (which can occur since $F$ does not satisfy the Jacobi identities; here there are none), in the depths up to `max_depth`. The cell below computes $\bar M(U)$ for each simple top space $U$, its maximal submodule $J$ and its simple quotient, and compares the simple quotient with the character of $\mathcal W(h)$, charge by charge. For the top spaces at $h = -\frac18, 1, \frac38$, $\bar M(U)$ is already simple: its maximal submodule is zero. For $h = 0$, the simple quotient of $\bar M(U)$ is the vacuum module $\mathcal W(0) = \mathcal W(2)$.

In [8]:
M = {}
for h, U in tops.items():
    D = 6 if U.dimension() == 1 else 5
    M[h] = P.induced_module_up_to(U, 8, D)
    J = M[h].maximal_submodule()
    L = M[h].simple_quotient()
    ok = all(L.weight_space(h + d, horizontal_grade=g).dimension() == chi(h, d, g) for d in range(D + 1) for g in charges)
    print(h, [L.weight_space(h + d).dimension() for d in range(D + 1)], ok,
          'J:', [J.weight_space(h + d).dimension() for d in range(D + 1)])

0 [1, 0, 1, 4, 5, 8, 10] True J: [0, 4, 4, 4, 8, 12, 24]


-1/8 [1, 1, 4, 5, 9, 13, 21] True J: [0, 0, 0, 0, 0, 0, 0]


1 [2, 2, 2, 4, 6, 12] True J: [0, 0, 0, 0, 0, 0]


3/8 [2, 2, 4, 6, 12, 16] True J: [0, 0, 0, 0, 0, 0]


## 5. The module $\mathcal W(0) \to 2\,\mathcal W(1)$

The module $\bar M(\mathbb C)$ induced from the trivial top space $\mathbb C$ is not simple: its maximal submodule has the character of $2\,\mathcal W(1)$. The maximal submodule is generated by the four vectors of depth 1, $T_{-1} v$, $W^+_{-1} v$, $W^0_{-1} v$ and $W^-_{-1} v$, where $v$ spans the top space. They span two copies of the doublet top space of $\mathcal W(1)$, with the charges $2, 0$ and $0, -2$: the charges $\pm 1$ of $\mathcal W(1)$, shifted by $+1$ and by $-1$. So $\bar M(\mathbb C) = \mathcal W(0) \to 2\,\mathcal W(1)$, the quotient of the projective cover $\mathcal P(0)$ by its socle. The cell below compares the dimensions of $\bar M(\mathbb C)$ with the character $\chi_{\mathcal W(0)} + 2\chi_{\mathcal W(1)}$, and checks that the submodule $N$ generated by the four vectors is the maximal submodule $J$, with the expected dimensions charge by charge.

In [9]:
M0 = M[0]
print([M0.weight_space(d).dimension() for d in range(7)])
print([sum(chi(0, d, g) + chi(1, d - 1, g - 1) + chi(1, d - 1, g + 1) for g in charges) for d in range(7)])
Tg, Wpg, W0g, Wmg = P.gens()
v = M0.highest_weight_vector()
N = M0.submodule([X.mode(v, -1) for X in (Tg, Wpg, W0g, Wmg)])
J = M0.maximal_submodule()
print([N.weight_space(d).dimension() for d in range(7)], [J.weight_space(d).dimension() for d in range(7)])
print(all(N.weight_space(d, horizontal_grade=g).dimension() == chi(1, d - 1, g - 1) + chi(1, d - 1, g + 1)
          for d in range(1, 7) for g in charges))

[1, 4, 5, 8, 13, 20, 34]
[1, 4, 5, 8, 13, 20, 34]
[0, 4, 4, 4, 8, 12, 24] [0, 4, 4, 4, 8, 12, 24]
True


## 6. The projective cover $\mathcal P(0)$

The Jordan block, on which $[T]$ acts by $\begin{pmatrix} 0 & 1 \\ 0 & 0\end{pmatrix}$, is the projective cover of the trivial module over the Zhu algebra, so the module $\bar M(U)$ induced from it is projective (`11_fusion_and_extensions.ipynb`, section 5). The cell below shows that $\bar M(U)$ has the character $2\chi_{\mathcal W(0)} + 2\chi_{\mathcal W(1)}$ of $\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0)$, charge by charge, and that $L_0 = T_0$ is not diagonalisable on its top space: for the basis $u_0, u_1$ of the top space, $L_0 u_1 = u_0$ and $L_0 u_0 = 0$. The vector $u_0$ generates the socle $\mathcal W(0)$, and the quotient by the socle is the module $\bar M(\mathbb C)$ of section 5.

In [10]:
Mj = P.induced_module_up_to(jordan, 8, 5)
print([Mj.weight_space(d).dimension() for d in range(6)])
print(all(Mj.weight_space(d, horizontal_grade=g).dimension() == 2*chi(0, d, g) + chi(1, d - 1, g - 1) + chi(1, d - 1, g + 1)
          for d in range(6) for g in charges))
top = Mj.weight_space(0).basis()
print(top, [Tg.mode(u, 0) for u in top])
soc = Mj.submodule([Tg.mode(top[1], 0)])
print(all(soc.weight_space(d, horizontal_grade=g).dimension() == chi(0, d, g) for d in range(6) for g in charges))
print([(Mj / soc).weight_space(d).dimension() for d in range(6)], [M0.weight_space(d).dimension() for d in range(6)])

[2, 4, 6, 12, 18, 28]
True
[u[0], u[1]] [0, u[0]]
True
[1, 4, 5, 8, 13, 20] [1, 4, 5, 8, 13, 20]


The same module can be built from the symplectic fermions themselves. The zero modes $\chi^\pm_0$ act on a Grassmann algebra, a top space $\mathbb C^{2|2}$ with basis $1, \theta^+, \theta^-, \theta^+\theta^-$ of charges $0, 1, -1, 0$. The module induced from it over the symplectic fermions is logarithmic, and the cell below checks that its even part has the character of $\mathcal P(0)$, charge by charge.

In [11]:
Ep = matrix(QQ, 4, 4, {(1, 0): 1, (3, 2): 1})
Em = matrix(QQ, 4, 4, {(2, 0): 1, (3, 1): -1})
Msf = SF.induced_module(SF.zhu_algebra().module({'cp': Ep, 'cm': Em}, parities=[0, 1, 1, 0], horizontal_grades=[0, 1, -1, 0]))
print(all(sum(1 for k in Msf.weight_space(d, horizontal_grade=g).basis_keys() if Msf._parity_key(k) == 0)
          == Mj.weight_space(d, horizontal_grade=g).dimension() for d in range(6) for g in charges))

True


## 7. Up to a weight and a depth

`induced_module_up_to(U, max_weight, max_depth)` only uses the relations of weight $\le$ `max_weight`, in the depths $\le$ `max_depth`. Everything it quotients by vanishes in $\bar M(U)$, so the result maps onto $\bar M(U)$ in these depths, and it can only shrink as `max_weight` and `max_depth` grow. The cell below computes $\bar M(\mathbb C)$ of section 5 for four pairs of bounds. Nothing changes from the relations of weight 6, the lowest, on, or from depth 4 on. This stability is evidence, not a proof (section 11).

In [12]:
for W, D in ((6, 4), (6, 5), (8, 4), (10, 5)):
    N = P.induced_module_up_to(tops[0], W, D)
    print(W, D, [N.weight_space(d).dimension() for d in range(D + 1)])

6 4 [1, 4, 5, 8, 13]


6 5 [1, 4, 5, 8, 13, 20]
8 4 [1, 4, 5, 8, 13]


10 5 [1, 4, 5, 8, 13, 20]


## 8. The projective cover $\mathcal P(1)$ through the level-1 Zhu algebra

The projective cover of $\mathcal W(1)$ has the structure $\mathcal P(1) = \mathcal W(1) \to 2\,\mathcal W(0) \to \mathcal W(1)$ [GK1996b]. Its lowest weight space has weight 0 and lies in the middle layer $2\,\mathcal W(0)$, so it does not generate $\mathcal P(1)$, and $\mathcal P(1)$ is not of the form $\bar M(U)$ of section 6. The head $\mathcal W(1)$ starts at depth 1. The level-$n$ Zhu algebra $A_n$ of Dong, Li and Mason [DLM1998] sees the weights $0, \dots, n$ of a module at once, and the universal module $\bar M_n(U)$, induced from an $A_n$-module $U$ placed in degree $n$, can have vectors in the degrees below $n$ as well as above (`11_fusion_and_extensions.ipynb`, section 5, and `09_zhu_algebras.ipynb`, section 11). Here $n = 1$.

The cell below computes the level-1 Zhu algebra $A_1(\mathcal W(2))$ from the presentation [He2017, BVY2019], with the relations up to weight 8 and the words of filtration up to 18 (`degree_bound=18`; the filtration of a word is the sum of the weights of its generators). This algebra maps onto $A_1(\mathcal W(2))$; it is finite-dimensional, of dimension 24.

In [13]:
Z1 = P.zhu_algebra_up_to(8, level=1, degree_bound=18)
print(Z1.dimension())

24


The modules of $A_1$ come from its structure as a finite-dimensional algebra over $\mathbb Q$: its radical $J$ (the kernel of the trace form), primitive idempotents $e$ (split off with minimal polynomials), the projective indecomposable modules $A_1 e$ and the simple modules $A_1 e / Je$, all with bases that are homogeneous for the charge grading. The cell below prints the dimension of the radical; then each simple module with the eigenvalue of $[T]$, its dimension and the dimension of its projective cover; then the Cartan matrix $c_{ij} = \dim e_i A_1 e_j$; and finally $\sum \dim S \cdot \dim P = \dim A_1$, with the sum over the simple modules $S$ and their projective covers $P$.

In [14]:
print(len(Z1.radical_basis()))
Ps, Ss = Z1.projective_indecomposable_modules(), Z1.simple_modules()
print([(S_.representation_matrix(Tg)[0, 0], S_.dimension(), P_.dimension()) for P_, S_ in zip(Ps, Ss)])
print(Z1.cartan_invariants_matrix())
print(sum(S_.dimension() * P_.dimension() for P_, S_ in zip(Ps, Ss)))

5


[(2, 2, 2), (11/8, 2, 2), (7/8, 1, 1), (3/8, 2, 2), (-1/8, 1, 1), (1, 2, 4), (0, 1, 2)]


[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 2 0]
[0 0 0 0 0 0 2]
24


There are seven simple $A_1$-modules: the top spaces of $\mathcal W(-\frac18)$, $\mathcal W(0)$, $\mathcal W(\frac38)$ and $\mathcal W(1)$, on which $[T]$ acts by $h$, and the degree-1 spaces of $\mathcal W(-\frac18)$, $\mathcal W(\frac38)$ and $\mathcal W(1)$, on which $[T]$ acts by $h + 1$; the degree-1 space of $\mathcal W(0)$ is zero. Only the simple modules with $[T] = 0$ and $[T] = 1$ have projective covers larger than themselves. On the projective cover of the simple module with $[T] = 1$, of dimension 4, $[T]$ has two Jordan blocks, as the cell below shows. The charges of this projective module are those of the words of $A_1$, which are fixed only up to a shift.

In [15]:
P1top = [P_ for P_, S_ in zip(Ps, Ss) if S_.representation_matrix(Tg)[0, 0] == 1][0]
print(P1top.representation_matrix(Tg).jordan_form(), P1top.horizontal_grades())

[1 1|0 0]
[0 1|0 0]
[---+---]
[0 0|1 1]
[0 0|0 1] [(-2), (-2), (0), (0)]


Inducing this projective module in degree 1, so that the lowest weight is $1 - 1 = 0$, gives $\mathcal P(1)$, here to depth 4. The cell below checks its character $2\chi_{\mathcal W(1)} + 2\chi_{\mathcal W(0)}$, charge by charge. In the function `p1(d, g)`, the two copies of $\mathcal W(1)$ carry the charges of $\mathcal W(1)$, and the two copies of $\mathcal W(0)$ carry the charges of $\mathcal W(0)$ shifted by $+1$ and by $-1$; the library's charges are those of `p1` shifted by $-1$.

In [16]:
D = 4
M1 = P.induced_module_up_to(P1top, 8, D)
print(M1._cover_module.top_weight(), [M1.weight_space(d).dimension() for d in range(D + 1)])


def p1(d, g):
    """The dimension of P(1) at weight d and charge g."""
    return 2*chi(1, d - 1, g) + chi(0, d, g - 1) + chi(0, d, g + 1)


print(all(M1.weight_space(d, horizontal_grade=(g - 1,)).dimension() == p1(d, g) for d in range(D + 1) for g in charges))

0 [2, 4, 6, 12, 18]
True


**Loewy layers.** The submodule generated by the weight-0 space is the radical $2\,\mathcal W(0) \to \mathcal W(1)$; the image of $L_0 - 1$ on the weight-1 space generates the socle $\mathcal W(1)$; and the quotient by the radical is the head $\mathcal W(1)$. The cell below computes the three, and compares them with the characters, charge by charge.

In [17]:
rad = M1.submodule(list(M1.weight_space(0).basis()))
soc = M1.submodule([x for x in (Tg.nproduct(u, 1) - u for u in M1.weight_space(1).basis()) if x])
print([rad.weight_space(d).dimension() for d in range(D + 1)], [soc.weight_space(d).dimension() for d in range(D + 1)],
      [(M1 / rad).weight_space(d).dimension() for d in range(D + 1)])
print(all(rad.weight_space(d, horizontal_grade=(g - 1,)).dimension() == chi(1, d - 1, g) + chi(0, d, g - 1) + chi(0, d, g + 1)
          and soc.weight_space(d, horizontal_grade=(g - 1,)).dimension() == chi(1, d - 1, g)
          for d in range(D + 1) for g in charges))

[2, 2, 4, 10, 14] [0, 2, 2, 2, 4] [0, 2, 2, 2, 4]


True


## 9. Fusion products

`M.fusion_product(N)` is the fusion product $M \boxtimes N$, computed by the Nahm–Gaberdiel–Kausch algorithm [Nah1994, GK1996] in the conventions of [KR2019], with $M$ at coordinate $w_1 = 1$ and $N$ at $w_2 = 0$ (`11_fusion_and_extensions.ipynb`, sections 1 and 2). What it computes is the depth-$d$ quotient $(M \boxtimes N)^{(d)}$, the quotient of $M \boxtimes N$ by the words of modes of total degree $> d$: a finite-dimensional space on which $L_0$ acts, with Jordan blocks for logarithmic fusion products. It is computed from the tensors of total depth $\le$ `max_depth`. Once Nahm's special subspace of $M$ fits in this window, the result maps onto the true quotient, and it only shrinks as the window grows.

The cell below computes the depth-0 quotients of the products of the four simple modules, with the generalised eigenvalues of $L_0$ and the sizes of its Jordan blocks.

In [18]:
Wsimple = {h: (M[h].simple_quotient() if h == 0 else M[h]) for h in tops}
hs = [0, 1, -1/8, 3/8]
for i, h1 in enumerate(hs):
    for h2 in hs[i:]:
        F = Wsimple[h1].fusion_product(Wsimple[h2]).depth_quotient_up_to(0, max_depth=3)
        print('W(%s) x W(%s):' % (h1, h2), F.conformal_weights())

W(0) x W(0): [(0, [1])]
W(0) x W(1): [(1, [1, 1])]
W(0) x W(-1/8): [(-1/8, [1])]
W(0) x W(3/8): [(3/8, [1, 1])]
W(1) x W(1): [(0, [1])]
W(1) x W(-1/8): [(3/8, [1, 1])]
W(1) x W(3/8): [(-1/8, [1])]
W(-1/8) x W(-1/8): [(0, [2])]
W(-1/8) x W(3/8): [(0, [1, 1]), (1, [1, 1])]
W(3/8) x W(3/8): [(0, [2])]


These are the fusion rules of Gaberdiel and Kausch [GK1996b]: $\mathcal W(0)$ is the unit; $\mathcal W(1) \boxtimes \mathcal W(1) = \mathcal W(0)$; fusion with $\mathcal W(1)$ exchanges $\mathcal W(-\frac18)$ and $\mathcal W(\frac38)$; $\mathcal W(-\frac18)^{\boxtimes 2}$ and $\mathcal W(\frac38)^{\boxtimes 2}$ have a Jordan block of $L_0$ at weight 0, since both are the logarithmic module $\mathcal P(0)$; and the depth-0 quotient of $\mathcal W(-\frac18) \boxtimes \mathcal W(\frac38) = \mathcal P(1)$ has the two states of weight 0 of the middle layer of $\mathcal P(1)$ and the two states of weight 1 of its head.

At depth 1, $\mathcal W(-\frac18) \boxtimes \mathcal W(\frac38)$ shows the Jordan blocks of $L_0$ at weight 1 of $\mathcal P(1)$, between its head and its socle $\mathcal W(1)$. The cell below prints the dimension of the depth-1 quotient for `max_depth` 1 to 4, and the conformal weights for `max_depth` 4. The window is large enough from `max_depth` 2 on. With `max_depth` 1, the window does not yet contain the special subspace of $\mathcal W(-\frac18)$, and the result is too small.

In [19]:
P1fus = Wsimple[-1/8].fusion_product(Wsimple[3/8])
print([P1fus.depth_quotient_up_to(1, max_depth=K).dimension() for K in (1, 2, 3, 4)])
print(P1fus.depth_quotient_up_to(1, max_depth=4).conformal_weights())

[6, 8, 8, 8]
[(0, [1, 1]), (1, [2, 2]), (2, [1, 1])]


The depth-0 quotient is a module over the Zhu algebra, as its zero modes satisfy the relations of $A(\mathcal W(2))$ (`11_fusion_and_extensions.ipynb`, section 1). For $\mathcal W(-\frac18) \boxtimes \mathcal W(-\frac18)$, the cell below shows that the depth-0 quotient is the Jordan block of section 6, in another basis, and that the module induced from it is $\mathcal P(0)$, charge by charge.

In [20]:
Ffus = Wsimple[-1/8].fusion_product(Wsimple[-1/8]).depth_quotient_up_to(0, max_depth=3)
Ufus = Ffus.zhu_algebra_module(Z)
print(Ufus.generator_matrices()['T'])
Mfus = P.induced_module_up_to(Ufus, 8, 5)
print([Mfus.weight_space(d).dimension() for d in range(6)])
print(all(Mfus.weight_space(d, horizontal_grade=g).dimension() == Mj.weight_space(d, horizontal_grade=g).dimension()
          for d in range(6) for g in charges))

[ -1/4 -1/16]
[    1   1/4]


[2, 4, 6, 12, 18, 28]
True


## 10. The same, by the library: simple modules, projective covers, Loewy layers, extensions

Sections 3–8 found the top spaces by solving the Gröbner basis on matrices, and chose the projective modules of $A_0$ and $A_1$ by hand. The library does all of this itself (with proofs in a companion paper, in preparation).

`P.simple_modules_up_to(max_weight, max_depth)` returns the simple quotients of the modules induced from the simple modules of the computed Zhu algebra $A_0$. The computed $A_0$ maps onto $A(\mathcal W(2))$, so every simple module is among them. They are sorted by lowest weight. The cell below prints the dimensions of their weight spaces and, for $\mathcal W(-\frac18)$ and $\mathcal W(0)$, compares them with the characters, charge by charge.

In [21]:
Simples = P.simple_modules_up_to(8, 5)
for S_ in Simples:
    h = S_._weight_base()
    print(h, [S_.weight_space(h + d).dimension() for d in range(6)],
          all(S_.weight_space(h + d, horizontal_grade=(g,)).dimension() == chi(h, d, g) for d in range(6) for g in charges)
          if h in (0, -1/8) else '')
Lm8, L0, L38, L1 = Simples

-1/8 [1, 1, 4, 5, 9, 13] True
0 [1, 0, 1, 4, 5, 8] True
3/8 [2, 2, 4, 6, 12, 16] 
1 [2, 2, 2, 4, 6, 12] 


For $\mathcal W(1)$ and $\mathcal W(\frac38)$, the library's charges of the top space are $0$ and $-2$ rather than $\pm 1$, since the grading of a module is fixed only up to a shift; this is why the cell above does not compare them charge by charge.

`L.projective_cover(max_weight, max_depth)` returns the projective cover $\pi: \mathcal P \to L$ (`11_fusion_and_extensions.ipynb`, section 5). Its domain $\mathcal P$ is $\bar M_n(A_n e)$, induced in degree $n$ from the projective indecomposable module of the level-$n$ Zhu algebra whose head is the top space of $L$. The level $n$ is chosen so that $h_L - n$ is the lowest weight in $h_L + \mathbb Z$ among the simple modules found from $A_0$: $n = 0$ for $\mathcal W(0)$, and $n = 1$ for $\mathcal W(1)$, since the lowest weight of $\mathcal W(0)$ is one less. The cell below computes the projective covers of $\mathcal W(0)$, $\mathcal W(1)$ and $\mathcal W(-\frac18)$, and compares them with the modules of sections 6 and 8; the projective simple module $\mathcal W(-\frac18)$ is its own projective cover.

In [22]:
import time
t = time.time()
pi0, pi1 = L0.projective_cover(8, 4), L1.projective_cover(8, 4)
P0, P1 = pi0.domain(), pi1.domain()
print('%.0f s' % (time.time() - t))
print([P0.weight_space(d).dimension() for d in range(5)], [Mj.weight_space(d).dimension() for d in range(5)])
print([P1.weight_space(d).dimension() for d in range(5)], [M1.weight_space(d).dimension() for d in range(5)])
Pm8 = Lm8.projective_cover(8, 4).domain()
print([Pm8.weight_space(-1/8 + d).dimension() for d in range(5)], [Lm8.weight_space(-1/8 + d).dimension() for d in range(5)])

13 s
[2, 4, 6, 12, 18] [2, 4, 6, 12, 18]
[2, 4, 6, 12, 18] [2, 4, 6, 12, 18]


[1, 1, 4, 5, 9] [1, 1, 4, 5, 9]


**Loewy layers.** `radical_series_up_to(depth)` (head first, with $\operatorname{rad} X$ the intersection of the kernels of all maps from $X$ to simple modules) and `socle_series_up_to(depth)` (socle first) give the layers as multiplicities of the simple modules that can occur, here $\mathcal W(0)$ and $\mathcal W(1)$, in this order (`11_fusion_and_extensions.ipynb`, section 5). The cell below shows that both series give $\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0)$ and $\mathcal P(1) = \mathcal W(1) \to 2\,\mathcal W(0) \to \mathcal W(1)$.

In [23]:
print(P0.radical_series_up_to(4), P0.socle_series_up_to(4))
print(P1.radical_series_up_to(4), P1.socle_series_up_to(4))

[[1, 0], [0, 2], [1, 0]] [[1, 0], [0, 2], [1, 0]]


[[0, 1], [2, 0], [0, 1]] [[0, 1], [2, 0], [0, 1]]


**$\operatorname{Ext}^1$ from the projective covers.** `M.ext1_up_to(N, depth, cover)` is $\operatorname{Hom}(K, N)$ modulo the restrictions of $\operatorname{Hom}(\mathcal P, N)$, where $K = \ker \pi$, computed up to `depth` (`11_fusion_and_extensions.ipynb`, section 5). The cell below shows that $\operatorname{Ext}^1$ between $\mathcal W(0)$ and $\mathcal W(1)$ is two-dimensional in both directions, as the multiplicities of the middle Loewy layers predict, and that the self-extensions of $\mathcal W(0)$ and of $\mathcal W(1)$ vanish. It also shows that $\operatorname{Ext}^1(\mathcal W(-\frac18), N) = 0$ for $N = \mathcal W(-\frac18), \mathcal W(\frac38)$, as $\mathcal W(-\frac18)$ is projective.

In [24]:
for L, pi in ((L0, pi0), (L1, pi1)):
    print([L.ext1_up_to(N_, 3, pi).dimension() for N_ in (L0, L1)])
print([Lm8.ext1_up_to(N_, 3, Lm8.projective_cover(8, 4)).dimension() for N_ in (Lm8, L38)])

[0, 2]


[2, 0]
[0, 0]


Each class $c$ gives an extension module, `extension(c)`: the pushout of the cover along the map $g: K \to N$ that represents $c$. The cell below builds an extension of $\mathcal W(0)$ by $\mathcal W(1)$ and one of $\mathcal W(1)$ by $\mathcal W(0)$, and prints their socle series. The socles are $\mathcal W(1)$ and $\mathcal W(0)$ alone, respectively, so neither extension splits.

In [25]:
E01 = L0.ext1_up_to(L1, 3, pi0).extension([1, 0])
E10 = L1.ext1_up_to(L0, 3, pi1).extension([1, 0])
for E in (E01, E10):
    print([E.weight_space(d).dimension() for d in range(4)], E.socle_series_up_to(3))

[1, 2, 3, 6] [[0, 1], [1, 0]]
[1, 2, 3, 6] [[1, 0], [0, 1]]


**Covers that are not projective.** The cover in `ext1_up_to` need not be projective. For a surjection $\bar M_n(U) \to M$ with kernel $K$, there is an exact sequence

$$0 \to \operatorname{Hom}(M, N) \to \operatorname{Hom}(\bar M_n(U), N) \to \operatorname{Hom}(K, N) \to \operatorname{Ext}^1(M, N) \xrightarrow{\ \delta\ } \operatorname{Ext}^1_{A_n}(U, N(n)).$$

So $\operatorname{Ext}^1(M, N)$ consists of the classes that come from $\operatorname{Hom}(K, N)$, which are those whose degree-$n$ part splits, together with classes that $\delta$ maps to nonzero classes of the Zhu algebra. `top_level_dimension()` is the dimension of $\operatorname{Ext}^1_{A_n}(U, N(n))$, and `delta_image()` is a basis of the image of $\delta$: the classes $u$ for which some module map $g: \tilde K \to N$ restricts to the inclusion of $N(n)$, where $\tilde K$ is the kernel of the module induced from the extension of $U$ of class $u$. `top_extension(u)` builds the extension module of a class $u$ that lifts.

The Jordan block of $[T]$ at 0 is a self-extension of the trivial $A_0$-module. In the cell below, with the cover $\bar M(\mathbb C) \to \mathcal W(0)$, this class does not lift to a self-extension of $\mathcal W(0)$, in agreement with $\operatorname{Ext}^1(\mathcal W(0), \mathcal W(0)) = 0$. With the identity of $\bar M(\mathbb C) = \mathcal W(0) \to 2\,\mathcal W(1)$ as the cover, the class lifts to an extension of $\bar M(\mathbb C)$ by $\mathcal W(0)$, and this extension is $\mathcal P(0)$, with the Loewy layers $\mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0)$.

In [26]:
W0s = M0.simple_quotient()
E = W0s.ext1_up_to(W0s, 3, Hom(M0, W0s).basis()[0])
print(E.top_level_dimension(), E.delta_image(), E.dimension())
F = M0.ext1_up_to(W0s, 3, End(M0).identity())
print(F.top_level_dimension(), F.delta_image(), F.dimension())
Pd = F.top_extension([1])
print([Pd.weight_space(d).dimension() for d in range(4)], Pd.radical_series_up_to(3))

1 [] 0
1 [(1,)] 1
[2, 4, 6, 12] [[1, 0], [0, 2], [1, 0]]


**Fusion of indecomposable modules.** Indecomposable modules are quotients of projective modules, and they can be fused. The quotient $\mathcal P(0) / \operatorname{rad}^2 \mathcal P(0)$ is $\mathcal W(0) \to 2\,\mathcal W(1)$, with character $\chi_{\mathcal W(0)} + 2\chi_{\mathcal W(1)}$. The cell below builds this quotient from the radical series and prints its dimensions, and then computes three depth-0 quotients:

* its product with the projective module $\mathcal W(-\frac18)$ shows the top spaces of $\mathcal W(-\frac18) \oplus 2\,\mathcal W(\frac38)$: one state of weight $-\frac18$ and four of weight $\frac38$;
* $\mathcal P(0) \boxtimes \mathcal W(-\frac18)$ shows the top spaces of $2\,\mathcal W(-\frac18) \oplus 2\,\mathcal W(\frac38)$;
* $\mathcal P(0) \boxtimes \mathcal P(0)$ shows the generating states of $2\,\mathcal P(0) \oplus 2\,\mathcal P(1)$, with two Jordan blocks at weight 0.

In [27]:
subs, layers = P0.radical_series_up_to(4, submodules=True)
R0 = P0.quotient(subs[2])
print([R0.weight_space(d).dimension() for d in range(5)])
for X, Y_ in ((R0, Lm8), (P0, Lm8), (P0, P0)):
    print(X.fusion_product(Y_).depth_quotient_up_to(0, 3).conformal_weights())

[1, 4, 5, 8, 13]
[(-1/8, [1]), (3/8, [1, 1, 1, 1])]


[(-1/8, [1, 1]), (3/8, [1, 1, 1, 1])]
[(0, [2, 2, 1, 1, 1, 1]), (1, [1, 1, 1, 1])]


## 11. Limitations

* **Larger triplet algebras.** $\mathcal W(p)$ for $p \ge 3$, a subalgebra of the lattice vertex algebra of $\sqrt{2p}\,\mathbb Z$ [AM2008], and the $(p_+, p_-)$ triplet algebras of `14_triplet_algebra.ipynb` are not treated. In particular, the projective cover of the vacuum module of the $(2, 3)$ triplet algebra at $c = 0$ [GRW2009] would need larger depths, and so faster module actions (in the C++ engines) and parallel linear algebra.
* **Twisted modules.** There are no twisted (Ramond) induced modules. Here the twisted sector of the symplectic fermions is reached through the untwisted modules $\mathcal W(-\frac18)$ and $\mathcal W(\frac38)$ of $\mathcal W(2)$ itself.
* **Up to a weight and a depth.** All modules are computed from the relations up to `max_weight` and in the depths up to `max_depth`, the fusion products in a window, and Loewy layers and $\operatorname{Ext}^1$ up to `depth`. The code cannot tell when these results have stabilised, and the agreement with the symplectic fermion characters is a check in the depths computed only.
* **Charges up to a shift.** The grading of an induced module is fixed only up to a shift (sections 8 and 10), so comparisons charge by charge need the shift to be found by hand.

## References

- [Kau1991] H. G. Kausch, *Extended conformal algebras generated by a multiplet of primary fields*, Phys. Lett. B 259 (1991), 448–455, [doi:10.1016/0370-2693(91)91655-F](https://doi.org/10.1016/0370-2693(91)91655-F). — the triplet algebras (introduction).
- [GK1996b] M. R. Gaberdiel, H. G. Kausch, *A rational logarithmic conformal field theory*, Phys. Lett. B 386 (1996), 131–137, [arXiv:hep-th/9606050](https://arxiv.org/abs/hep-th/9606050). — the modules of $\mathcal W(2)$, the projective modules $\mathcal P(0)$ and $\mathcal P(1)$, and the fusion rules (introduction, sections 8, 9).
- [Abe2007] T. Abe, *A $\mathbb Z_2$-orbifold model of the symplectic fermionic vertex operator superalgebra*, Math. Z. 255 (2007), 755–792, [arXiv:math/0503472](https://arxiv.org/abs/math/0503472). — $\mathcal W(2)$ as the even part of symplectic fermions, $C_2$-cofinite and not rational (introduction).
- [AM2008] D. Adamović, A. Milas, *On the triplet vertex algebra $\mathcal W(p)$*, Adv. Math. 217 (2008), 2664–2699, [arXiv:0707.1857](https://arxiv.org/abs/0707.1857). — $\mathcal W(p)$ is $C_2$-cofinite and not rational, and its Zhu algebra (introduction, sections 3, 11).
- [Kau2000] H. G. Kausch, *Symplectic fermions*, Nuclear Phys. B 583 (2000), 513–541, [arXiv:hep-th/0003029](https://arxiv.org/abs/hep-th/0003029). — symplectic fermions and their twisted sector (introduction, section 1).
- [Zhu1996] Y. Zhu, *Modular invariance of characters of vertex operator algebras*, J. Amer. Math. Soc. 9 (1996), 237–302, [doi:10.1090/S0894-0347-96-00182-8](https://doi.org/10.1090/S0894-0347-96-00182-8). — the Zhu algebra (section 3).
- [DLM1998] C. Dong, H. Li, G. Mason, *Vertex operator algebras and associative algebras*, J. Algebra 206 (1998), 67–96, [arXiv:q-alg/9612010](https://arxiv.org/abs/q-alg/9612010). — the universal modules $\bar M(U)$ and $\bar M_n(U)$, and the level-$n$ Zhu algebras (sections 4, 8).
- [He2017] X. He, *Higher level Zhu algebras are subquotients of universal enveloping algebras*, J. Algebra 491 (2017), 265–279, [arXiv:1712.05671](https://arxiv.org/abs/1712.05671). — the model of the level-$n$ Zhu algebras used (section 8).
- [BVY2019] K. Barron, N. Vander Werf, J. Yang, *Higher level Zhu algebras and modules for vertex operator algebras*, J. Pure Appl. Algebra 223 (2019), 3295–3317, [arXiv:1710.04767](https://arxiv.org/abs/1710.04767). — higher level Zhu algebras and their modules (section 8).
- [Nah1994] W. Nahm, *Quasi-rational fusion products*, Internat. J. Modern Phys. B 8 (1994), 3693–3702, [arXiv:hep-th/9402039](https://arxiv.org/abs/hep-th/9402039). — the depth quotients and the special subspace (section 9).
- [GK1996] M. R. Gaberdiel, H. G. Kausch, *Indecomposable fusion products*, Nuclear Phys. B 477 (1996), 293–318, [arXiv:hep-th/9604026](https://arxiv.org/abs/hep-th/9604026). — the Nahm–Gaberdiel–Kausch algorithm (section 9).
- [KR2019] S. Kanade, D. Ridout, *NGK and HLZ: fusion for physicists and mathematicians*, in: Affine, vertex and W-algebras, Springer INdAM Ser. 37 (2019), 135–181, [arXiv:1812.10713](https://arxiv.org/abs/1812.10713). — the conventions of the fusion computation (section 9).
- [GRW2009] M. R. Gaberdiel, I. Runkel, S. Wood, *Fusion rules and boundary conditions in the c = 0 triplet model*, J. Phys. A 42 (2009), 325403, [arXiv:0905.0916](https://arxiv.org/abs/0905.0916). — the $(2,3)$ triplet algebra at $c = 0$ (section 11).